# BomaTrack | Next-day electricity forecast
**ICS 3202 · Ridge regression.** Select a date in the app. Seven complete daily meter readings go in; one estimate of tomorrow’s electricity use comes out, in kilowatt-hours (kWh). The data comes from one home in Sceaux, France. The earlier base notebook predicted invoice lateness; this final model predicts energy use.

**Run:** Use Python 3.11, NumPy 2.2.6 and scikit-learn 1.6.1. Place the [meter archive](https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip) next to this notebook, or set `BOMA_UCI_ZIP` to its path. Run the cells in order.

**Data:** Hébrail & Bérard, [UCI Individual Household Electric Power Consumption](https://doi.org/10.24432/C58K54), CC BY 4.0.

In [1]:
import csv, hashlib, io, json, os, tempfile, zipfile
from datetime import date, timedelta
from pathlib import Path
from functools import lru_cache
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

archive = Path(os.environ.get("BOMA_UCI_ZIP", "household_power_consumption.zip"))
if not archive.is_file() and "BOMA_UCI_ZIP" not in os.environ:
    archive = Path(tempfile.gettempdir()) / "BomaTrack_UCI_electricity.zip"
if not archive.is_file():
    raise FileNotFoundError("Download the meter archive linked above and place it next to the notebook.")
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest == "9f84b46ade8a2d8e1286ec4b2b6c2987a45a755c59f263be3b3b3d10dfbda3ff", "Check the UCI archive version"
day = timedelta(days=1)
out = Path("exports")
out.mkdir(exist_ok=True)

### 1. Make daily meter readings
Each row covers one minute. Divide its power in kW by 60 to get energy in kWh, then add the minutes for each date. Keep a day only if all 1,440 minutes are present exactly once with valid readings. A missing reading is not zero.

In [2]:
@lru_cache(maxsize=2048)
def parse_date(s):
    d,m,y = map(int,s.split("/"))
    return date(y,m,d)

totals, seen, counts, duplicates = {}, {}, {}, 0
bad, rows = 0, 0
with zipfile.ZipFile(archive) as z, z.open("household_power_consumption.txt") as f:
    reader = csv.reader(io.TextIOWrapper(f,encoding="ascii"),delimiter=";")
    assert next(reader)[:3] == ["Date","Time","Global_active_power"]
    for ds, clock, raw, *_ in reader:
        d = parse_date(ds)
        if d not in seen:
            seen[d], counts[d], totals[d] = bytearray(1440), 0, 0.0
        minute = int(clock[:2])*60 + int(clock[3:5])
        assert clock.endswith(":00") and 0 <= minute < 1440
        rows += 1
        if seen[d][minute]:
            duplicates += 1
            seen[d][minute] = 2
        else:
            seen[d][minute] = 1
        if raw in ("","?"):
            bad += 1
            continue
        kw = float(raw)
        assert np.isfinite(kw) and kw >= 0
        totals[d] += kw / 60
        counts[d] += 1
daily = {d:kwh for d,kwh in totals.items() if counts[d]==1440 and all(n==1 for n in seen[d])}
print(f"1. {rows:,} minute records across {len(seen):,} recorded dates ({min(seen)} to {max(seen)}).")
print(f"2. {bad:,} active-power readings missing; {duplicates} duplicate minute keys.")
print(f"3. {len(daily):,} complete days; {len(seen)-len(daily)} incomplete days excluded.")
print(f"4. Complete daily use ranges from {min(daily.values()):.2f} to {max(daily.values()):.2f} kWh (peaks retained).")

1. 2,075,259 minute records across 1,442 recorded dates (2006-12-16 to 2010-11-26).
2. 25,979 active-power readings missing; 0 duplicate minute keys.
3. 1,358 complete days; 84 incomplete days excluded.
4. Complete daily use ranges from 4.17 to 79.56 kWh (peaks retained).


### 2. Pair each week with the following day
For each target date, take the previous seven calendar days, oldest first. The target day’s reading is the answer, never an input. Train on the earliest dates, choose the model on 2009 dates, and save 2010 dates for the final test.

In [3]:
dates = sorted(d for d in daily if all(d-i*day in daily for i in range(7,0,-1)))
X = np.array([[daily[d-i*day] for i in range(7,0,-1)] for d in dates])
y = np.array([daily[d] for d in dates])
year = np.array([d.year for d in dates])
train, val, test = year <= 2008, year == 2009, year == 2010
assert (train.sum(),val.sum(),test.sum()) == (467,232,210)
print(f"5. Requiring a complete previous week leaves {len(dates)} eligible targets; {len(daily)-len(dates)} otherwise complete targets drop out.")
for name,mask in [("Train",train),("Validation",val),("Test",test)]:
    selected=[d for d,ok in zip(dates,mask) if ok]
    print(name,len(selected),selected[0],selected[-1])

5. Requiring a complete previous week leaves 909 eligible targets; 449 otherwise complete targets drop out.
Train 467 2006-12-29 2008-12-31
Validation 232 2009-01-01 2009-12-31
Test 210 2010-01-01 2010-11-25


### 3. Train and choose a model
Ridge learns a weight for each day. It puts the readings on a common scale and discourages very large weights. Try three penalty settings and a small decision tree. Choose the lowest **mean absolute error (MAE)**: the average size of a miss, in kWh.

In [4]:
def candidates():
    models = {}
    for penalty in (0.1, 1, 10):
        models[f"ridge-{penalty}"] = make_pipeline(StandardScaler(), Ridge(alpha=penalty))
    models["tree-depth3"] = DecisionTreeRegressor(max_depth=3, random_state=3102)
    return models

def scores(obs, pred):
    pred = np.maximum(0, pred)  # Electricity use cannot be negative.
    return {"MAE_kWh": float(mean_absolute_error(obs, pred)),
            "RMSE_kWh": float(root_mean_squared_error(obs, pred))}

valid = {}
for name, model in candidates().items():
    model.fit(X[train], y[train])  # Scaling is fitted on these rows too.
    valid[name] = scores(y[val], model.predict(X[val]))
    print(f"{name:15}  average miss: {valid[name]['MAE_kWh']:.2f} kWh")
chosen = min(valid, key=lambda name: valid[name]["MAE_kWh"])
print("Selected:", chosen)

ridge-0.1        average miss: 4.38 kWh
ridge-1          average miss: 4.38 kWh
ridge-10         average miss: 4.37 kWh
tree-depth3      average miss: 5.00 kWh
Selected: ridge-10


### 4. Test on later dates
Refit using the training and validation dates together. On the same 210 unseen dates, compare Ridge with a tree, repeating yesterday, and repeating the same weekday last week. A smaller average miss is better.

In [5]:
assert chosen.startswith("ridge-"), "The Android export below expects the selected model to be Ridge."
models = candidates()
development = train | val
predictions = {"Yesterday": X[test, -1], "Same weekday last week": X[test, 0]}
for name in (chosen, "tree-depth3"):
    models[name].fit(X[development], y[development])
    predictions[name] = models[name].predict(X[test])

final = {}
for name, pred in predictions.items():
    final[name] = scores(y[test], pred)
    print(f"{name:23}  average miss: {final[name]['MAE_kWh']:.2f} kWh")
winner = min(final, key=lambda name: final[name]["MAE_kWh"])
with open("heldout_predictions.csv","w",newline="",encoding="utf8") as f:
    w=csv.writer(f); w.writerow(["target_date","actual_kWh",*predictions])
    w.writerows(zip(np.array(dates)[test],y[test],*(np.maximum(0,p) for p in predictions.values())))

Yesterday                average miss: 5.12 kWh


Same weekday last week   average miss: 5.85 kWh
ridge-10                 average miss: 4.46 kWh
tree-depth3              average miss: 4.81 kWh


### 5. Save the model for Android
The phone needs two files: `french_ridge.properties` holds the model’s weights and scaling values; `french_daily.csv` holds the dated meter readings. The app uses them to calculate a new forecast when the user taps **Predict**.

In [6]:
model = models[chosen]
scaler = model.named_steps["standardscaler"]
ridge = model.named_steps["ridge"]
version="sceaux-ridge-v1"
fields={"version":version,"model":"ridge","units":"kWh","source_doi":"10.24432/C58K54","dataset_sha256":digest,
        "features":",".join(f"target_minus_{i}" for i in range(7,0,-1)),"output_floor":"0",
        "alpha":str(ridge.alpha),"intercept":repr(float(ridge.intercept_))}
for k,array in [("mean",scaler.mean_),("scale",scaler.scale_),("weights",ridge.coef_)]:
    fields[k]=",".join(repr(float(x)) for x in array)
def days_between(start,end):
    return (start+i*day for i in range((end-start).days+1))
test_dates=[d for d,ok in zip(dates,test) if ok]
targets=[test_dates[0],test_dates[len(test_dates)//2],test_dates[-1]]
gap=next(t-day for t in days_between(test_dates[0],test_dates[-1]) if t in daily and t not in test_dates)
fields["issue_shortcuts"]=",".join(str(t-day) for t in targets)+","+str(gap)
(out/"french_ridge.properties").write_text("".join(f"{k}={v}\n" for k,v in fields.items()),encoding="ascii")
with (out/"french_daily.csv").open("w",newline="",encoding="ascii") as f:
    w=csv.writer(f,lineterminator="\n"); w.writerow(["date","measured_kWh"])
    for d in days_between(test_dates[0]-7*day,test_dates[-1]):
        w.writerow([d,repr(daily[d]) if d in daily else ""])
print("Saved the model and daily meter readings in exports/.")

Saved the model and daily meter readings in exports/.


### 6. Check the app examples
Use the first, middle and last eligible test dates. For the first two, rebuild the estimate from the exported weights. Then check a week with missing data.

In [7]:
checks=[]
for t in targets:
    n=dates.index(t); pred=float(max(0,model.predict(X[n:n+1])[0]))
    checks.append(dict(issue=str(t-day),target=str(t),history=X[n].tolist(),prediction=pred,actual=float(y[n]),
                       difference=float(y[n])-pred))
report=dict(version=version,archive_sha256=digest,selected=chosen,validation=valid,test=final,
            winner_on_test=winner,examples=checks,gap_issue=str(gap),split=[int(m.sum()) for m in (train,val,test)])
Path("evaluation.json").write_text(json.dumps(report,indent=2)+"\n",encoding="utf8")
base = float(fields["intercept"])
means, spreads, weights = (np.fromstring(fields[key], sep=",") for key in ("mean", "scale", "weights"))
for case in checks[:2]:
    changes = (np.asarray(case["history"]) - means) / spreads * weights
    rebuilt = max(0.0, base + float(changes.sum()))
    assert np.isclose(rebuilt, case["prediction"])
    print(f"{case['target']}: forecast {rebuilt:.2f} kWh; meter {case['actual']:.2f} kWh.")
    print(f"  Latest input: {case['history'][-1]:.2f} kWh. Its adjustment to the forecast: {changes[-1]:+.2f} kWh.")
gap_day = date.fromisoformat(report["gap_issue"])
missing = [d for d in days_between(gap_day - 6*day, gap_day) if d not in daily]
assert missing
print("Insufficient history for", gap_day + day, "because a complete daily reading is missing on", *missing)

2010-01-01: forecast 34.77 kWh; meter 20.40 kWh.
  Latest input: 36.86 kWh. Its adjustment to the forecast: +3.38 kWh.
2010-06-26: forecast 22.86 kWh; meter 16.36 kWh.
  Latest input: 18.32 kWh. Its adjustment to the forecast: -2.32 kWh.
Insufficient history for 2010-01-03 because a complete daily reading is missing on 2010-01-02


**Result:** Ridge missed by **4.46 kWh on average**, compared with **5.12 kWh** for repeating yesterday. June’s latest reading was lower than December’s, pulling its forecast down rather than up. Individual errors can still be large. These results describe this household only.